# Data Reconciliation: Missing Rows Recovery

## Purpose
This notebook identifies commits that were removed from `apachejit_total.csv` but are missing from the enriched `apachejit_with_diffs_v2.csv` file. It then:
1. Adds the missing rows to the enriched file with empty placeholders
2. Extracts textual commit diffs from local cloned repositories
3. Saves the corrected, complete dataset

## Problem Context
During dataset preparation, some rows were removed from `apachejit_total.csv`. Later, `apachejit_with_diffs_v2.csv` was created to add textual commit diff features. However, this enriched file is missing rows that are present in the original total dataset. This notebook fixes the data integrity issue.

## Section 1: Load Both Datasets and Identify Missing Rows

In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from git import Repo
import warnings
warnings.filterwarnings('ignore')

# ======= FILE PATHS =======
TOTAL_CSV = '../../data/apachejit/apachejit_total.csv'
ENRICHED_CSV = '../../data/apachejit/apachejit_with_diffs_v2.csv'
LOCAL_REPO_BASE = '../../repos/apache/'
OUTPUT_CSV = '../../data/apachejit/apachejit_with_diffs_v2_reconciled.csv'
# =========================

# Load both datasets
print("Loading datasets...")
df_total = pd.read_csv(TOTAL_CSV)
df_enriched = pd.read_csv(ENRICHED_CSV)

print(f"\n📊 Dataset Overview:")
print(f"   Original total.csv:           {len(df_total):,} rows")
print(f"   Enriched with_diffs_v2.csv:   {len(df_enriched):,} rows")
print(f"   Difference:                    {len(df_total) - len(df_enriched):,} rows potentially missing")

# Display column information
print(f"\n📋 Columns in original (total.csv):")
print(f"   {len(df_total.columns)} columns: {list(df_total.columns)[:5]}... (and {len(df_total.columns)-5} more)")

print(f"\n📋 Columns in enriched (with_diffs_v2.csv):")
print(f"   {len(df_enriched.columns)} columns: {list(df_enriched.columns)[:5]}... (and {len(df_enriched.columns)-5} more)")

In [ ]:
# Identify missing rows using commit_id as unique identifier
print("\n🔍 Identifying missing rows using commit_id as key...\n")

# Create sets of commit_ids
total_commits = set(df_total['commit_id'].unique())
enriched_commits = set(df_enriched['commit_id'].unique())

# Find commits in total but not in enriched
missing_commits = total_commits - enriched_commits

print(f"Total unique commits in original:  {len(total_commits):,}")
print(f"Total unique commits in enriched:  {len(enriched_commits):,}")
print(f"Missing commits (in total, NOT in enriched): {len(missing_commits):,}")

# For commits that appear in both, check for duplicate row counts
if len(enriched_commits) > 0:
    rows_in_enriched_per_commit = df_enriched.groupby('commit_id').size()
    max_rows_per_commit = rows_in_enriched_per_commit.max()
    avg_rows_per_commit = rows_in_enriched_per_commit.mean()
    print(f"\nEnriched CSV - rows per commit: avg={avg_rows_per_commit:.2f}, max={max_rows_per_commit}")

# Display some example missing commits
if len(missing_commits) > 0:
    print(f"\n📝 First 10 missing commit IDs:")
    for i, commit_id in enumerate(list(missing_commits)[:10]):
        proj = df_total[df_total['commit_id'] == commit_id]['project'].iloc[0]
        print(f"   {i+1}. {commit_id[:8]}... ({proj})")
else:
    print("\n✅ No missing commits found - datasets are already aligned!")

## Section 2: Create Placeholder Rows for Missing Commits

In [ ]:
print("Creating placeholder rows for missing commits...\n")

# Get rows for all missing commits from the original dataset
df_missing = df_total[df_total['commit_id'].isin(missing_commits)].copy()

print(f"✓ Extracted {len(df_missing):,} rows for missing commits from original dataset")
print(f"  Projects involved: {df_missing['project'].nunique()} unique projects")
print(f"\n  Project distribution of missing rows:")
project_dist = df_missing['project'].value_counts()
for proj, count in project_dist.head(10).items():
    print(f"    • {proj}: {count} commits")

if len(project_dist) > 10:
    print(f"    ... and {len(project_dist) - 10} more projects")

# Initialize diff-related columns with NaN/empty values
# These will be populated later by the diff extraction step
print(f"\nInitializing diff-related columns with empty values...")

diff_related_cols = ['diff_text']  # Core diff column
# Add any other enriched columns from enriched CSV that don't exist in missing rows
for col in df_enriched.columns:
    if col not in df_missing.columns and col != 'diff_text':
        # Check if it's a metrics/feature column that should be empty initially
        if col.startswith(('cmplx_', 'graph_', 'tfidf_', 'approx_h_', 'km_', 'db_', 'hdb_', 'agg_')):
            diff_related_cols.append(col)

# Add missing columns to the dataframe with NaN
for col in diff_related_cols:
    if col not in df_missing.columns:
        df_missing[col] = np.nan

print(f"✓ Added {len(diff_related_cols)} enrichment columns: {diff_related_cols}")

# Align column order to match enriched CSV
# Put base columns first, then enrichment columns
base_cols = [c for c in df_enriched.columns if c not in diff_related_cols and c in df_missing.columns]
all_cols = base_cols + diff_related_cols
# Add any missing columns at the end
remaining_cols = [c for c in df_missing.columns if c not in all_cols]
all_cols = all_cols + remaining_cols

df_missing = df_missing[all_cols]

print(f"\n✓ Placeholder dataframe ready: {df_missing.shape}")
print(f"  Columns: {len(df_missing.columns)}")
print(f"  Rows: {len(df_missing):,}")

## Section 3: Merge Missing Rows into the Enriched Dataset

In [ ]:
print("Merging missing rows into enriched dataset...\n")

# Ensure both dataframes have the same columns in the same order
# First, collect all columns from both dataframes
all_columns = list(set(df_enriched.columns) | set(df_missing.columns))

# Reorder: base columns first, then enrichment columns
base_cols = ['commit_id', 'project', 'author_date', 'buggy', 'fix']
base_cols = [c for c in base_cols if c in all_columns]
other_cols = [c for c in all_columns if c not in base_cols]

all_columns = base_cols + sorted(other_cols)

# Align both dataframes to have the same columns
for df in [df_enriched, df_missing]:
    for col in all_columns:
        if col not in df.columns:
            df[col] = np.nan
    # Reorder columns
    df_temp = df[all_columns]
    if df is df_enriched:
        df_enriched = df_temp
    else:
        df_missing = df_temp

print(f"Column alignment complete: {len(all_columns)} columns")

# Concatenate the two dataframes
df_combined = pd.concat([df_enriched, df_missing], ignore_index=False, sort=False)

print(f"\n✓ Merged datasets:")
print(f"   Enriched rows:  {len(df_enriched):,}")
print(f"   Missing rows:   {len(df_missing):,}")
print(f"   Total combined: {len(df_combined):,}")

# Sort by commit_id for consistency (if it exists)
if 'commit_id' in df_combined.columns:
    df_combined = df_combined.sort_values('commit_id').reset_index(drop=True)
    print(f"\n✓ Sorted by commit_id for consistency")

# Verify no duplicates were introduced
duplicate_commits = df_combined['commit_id'].value_counts()
duplicates = duplicate_commits[duplicate_commits > 1]
if len(duplicates) > 0:
    print(f"\n⚠️  WARNING: Found {len(duplicates)} commits with multiple rows:")
    for commit_id, count in duplicates.head().items():
        print(f"    {commit_id[:8]}... appears {count} times")
else:
    print(f"\n✓ No duplicate commits - all rows are unique")

print(f"\nFinal combined dataset: {df_combined.shape}")
print(f"  Rows: {len(df_combined):,}")
print(f"  Columns: {len(df_combined.columns)}")

## Section 4: Extract Diffs for Newly Added Rows

In [ ]:
print("Building project-to-local-repo mapping...\n")

# ======= CONFIG =======
SAVE_EVERY = 10000  # save after extracting this many diffs
# ======================

# Dynamically discover local repositories
repo_mapping = {}
if os.path.exists(LOCAL_REPO_BASE):
    for folder in os.listdir(LOCAL_REPO_BASE):
        full_path = os.path.join(LOCAL_REPO_BASE, folder)
        if os.path.isdir(full_path) and os.path.isdir(os.path.join(full_path, '.git')):
            # Map both 'folderName' and 'apache/folderName' to the path
            repo_mapping[folder] = full_path
            repo_mapping[f'apache/{folder}'] = full_path

print(f"✓ Found {len(repo_mapping)//2} local repositories:")
for folder in sorted(set(repo_mapping.keys())):
    if '/' not in folder:  # Show only the base folder names (not the apache/ variants)
        print(f"   • {folder}")

# Helper function: compute diff for a commit
def compute_diff(repo, commit_sha):
    """Return unified diff of a commit against its first parent."""
    try:
        commit = repo.commit(commit_sha)
        if not commit.parents:
            return ""  # initial commit - no diff
        parent = commit.parents[0]
        diffs = parent.diff(commit, create_patch=True)
        parts = []
        for d in diffs:
            try:
                parts.append(d.diff.decode('utf-8', errors='ignore'))
            except Exception:
                parts.append(str(d.diff))
        return '\n'.join(parts)
    except Exception as e:
        return ""

# Identify rows that need diffs (missing rows with empty diff_text)
print(f"\nIdentifying rows that need diff extraction...\n")

# These are the rows we added (from df_missing) - they all have empty/NaN diff_text
rows_needing_diffs = df_combined[(df_combined['commit_id'].isin(missing_commits)) & 
                                  ((df_combined['diff_text'].isna()) | (df_combined['diff_text'] == ''))].copy()

print(f"✓ Rows requiring diff extraction: {len(rows_needing_diffs):,}")
print(f"   Out of {len(df_missing):,} newly added rows")

# Cache repo objects to avoid reopening
repo_objects = {}

# Process each row
print(f"\nExtracting diffs from local repositories...")
print(f"This will save progress every {SAVE_EVERY} commits.\n")

save_counter = 0
failed_repos = set()
successful_extractions = 0
failed_extractions = 0

for idx in tqdm(rows_needing_diffs.index, desc="Extracting local diffs"):
    row = df_combined.loc[idx]
    project = row['project']
    commit_id = row['commit_id']
    
    # Find local repo
    repo_path = repo_mapping.get(project)
    if not repo_path:
        # Try with just the project name (without apache/)
        repo_path = repo_mapping.get(project.split('/')[-1])
    
    if not repo_path:
        # Repository not found locally
        df_combined.at[idx, 'diff_text'] = ""
        failed_extractions += 1
        if project not in failed_repos:
            failed_repos.add(project)
        continue
    
    # Load repo (cached)
    if repo_path not in repo_objects:
        try:
            repo_objects[repo_path] = Repo(repo_path)
        except Exception as e:
            df_combined.at[idx, 'diff_text'] = ""
            failed_extractions += 1
            if project not in failed_repos:
                failed_repos.add(project)
            continue
    
    # Compute diff safely
    try:
        diff = compute_diff(repo_objects[repo_path], commit_id)
        df_combined.at[idx, 'diff_text'] = diff
        if diff:
            successful_extractions += 1
        else:
            failed_extractions += 1
    except Exception:
        df_combined.at[idx, 'diff_text'] = ""
        failed_extractions += 1
    
    save_counter += 1
    if save_counter >= SAVE_EVERY:
        # Save progress to avoid data loss
        df_combined.to_csv(OUTPUT_CSV, index=False)
        print(f"   💾 Progress saved: {successful_extractions} successful, {failed_extractions} failed")
        save_counter = 0

# Final save
df_combined.to_csv(OUTPUT_CSV, index=False)

print(f"\n✓ Diff extraction complete!")
print(f"   Successfully extracted: {successful_extractions:,} diffs")
print(f"   Failed to extract:      {failed_extractions:,} diffs")
if failed_repos:
    print(f"   Repositories with issues: {', '.join(sorted(failed_repos)[:5])}")
    if len(failed_repos) > 5:
        print(f"      ... and {len(failed_repos)-5} more")

## Section 5: Verify and Save the Corrected Dataset

In [ ]:
# Reload the combined dataset (in case we need to work with it fresh)
df_final = pd.read_csv(OUTPUT_CSV)

print("=" * 80)
print("DATA RECONCILIATION VERIFICATION")
print("=" * 80)

# Verification 1: All original commits present
print("\n✓ Commit Completeness Check:")
final_commits = set(df_final['commit_id'].unique())
all_should_have = set(df_total['commit_id'].unique())

if final_commits == all_should_have:
    print(f"   ✅ SUCCESS: All {len(final_commits):,} commits from original are now present")
else:
    still_missing = all_should_have - final_commits
    print(f"   ❌ ISSUE: Still missing {len(still_missing):,} commits")
    print(f"      Missing commit IDs: {list(still_missing)[:5]}")

# Verification 2: Diff extraction success rate
print("\n✓ Diff Extraction Coverage:")
with_diffs = (df_final['diff_text'].notna()) & (df_final['diff_text'] != '')
n_with_diffs = with_diffs.sum()
n_total = len(df_final)
coverage = 100 * n_with_diffs / n_total

print(f"   Rows with diffs:    {n_with_diffs:,} ({coverage:.1f}%)")
print(f"   Rows without diffs: {n_total - n_with_diffs:,} ({100-coverage:.1f}%)")

# For missing rows specifically
missing_with_diffs = with_diffs[df_final['commit_id'].isin(missing_commits)].sum()
missing_total = len(df_final[df_final['commit_id'].isin(missing_commits)])
if missing_total > 0:
    missing_coverage = 100 * missing_with_diffs / missing_total
    print(f"\n   For newly added {missing_total:,} rows:")
    print(f"      With diffs: {missing_with_diffs:,} ({missing_coverage:.1f}%)")
    print(f"      Without:    {missing_total - missing_with_diffs:,} ({100-missing_coverage:.1f}%)")

# Verification 3: Diff text statistics
print("\n✓ Diff Text Statistics:")
diff_lengths = df_final.loc[with_diffs, 'diff_text'].str.len()
if len(diff_lengths) > 0:
    print(f"   Mean length:   {diff_lengths.mean():.0f} characters")
    print(f"   Median length: {diff_lengths.median():.0f} characters")
    print(f"   Min length:    {diff_lengths.min():.0f} characters")
    print(f"   Max length:    {diff_lengths.max():.0f} characters")
    print(f"   Std dev:       {diff_lengths.std():.0f} characters")

# Verification 4: Data integrity - no duplicates
print("\n✓ Data Integrity Checks:")
duplicate_commits = df_final['commit_id'].value_counts()
n_duplicates = (duplicate_commits > 1).sum()
if n_duplicates == 0:
    print(f"   ✅ No duplicate commits found - all {len(df_final):,} rows are unique")
else:
    print(f"   ⚠️  Found {n_duplicates} commits with multiple rows")
    duplicates_list = duplicate_commits[duplicate_commits > 1]
    for commit_id, count in duplicates_list.head().items():
        print(f"      {commit_id[:8]}... appears {count} times")

# Verification 5: Column completeness
print("\n✓ Column Completeness:")
base_cols = ['commit_id', 'project', 'author_date', 'buggy']
missing_base_cols = [c for c in base_cols if c not in df_final.columns]
if missing_base_cols:
    print(f"   ⚠️  Missing base columns: {missing_base_cols}")
else:
    print(f"   ✅ All {len(base_cols)} base columns present")

enrichment_cols = [c for c in df_final.columns if c not in base_cols and c != 'fix']
if len(enrichment_cols) > 0:
    print(f"   ✅ Found {len(enrichment_cols)} enrichment columns")

# Final summary
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"\n📊 Final Dataset Statistics:")
print(f"   Total rows:           {len(df_final):,}")
print(f"   Total columns:        {len(df_final.columns)}")
print(f"   Unique commits:       {len(final_commits):,}")
print(f"   Rows with diffs:      {n_with_diffs:,} ({coverage:.1f}%)")
print(f"   Originally missing:   {len(missing_commits):,}")
print(f"   Successfully added:   {len(df_final[df_final['commit_id'].isin(missing_commits)]):,}")
print(f"\n✓ Output saved to: {OUTPUT_CSV}")
print(f"\n✅ Data reconciliation complete!")

## Appendix: Detailed Comparison Between Versions

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("Generating comparison visualizations...\n")

# Comparison of datasets
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Row count comparison
ax = axes[0, 0]
datasets = ['Original\n(apachejit_total.csv)', 'Enriched\n(apachejit_with_diffs_v2.csv)', 
            'Reconciled\n(apachejit_with_diffs_v2_reconciled.csv)']
row_counts = [len(df_total), len(df_enriched), len(df_final)]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
bars = ax.bar(datasets, row_counts, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Number of Rows', fontsize=11)
ax.set_title('Dataset Size Comparison', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
# Add value labels on bars
for bar, count in zip(bars, row_counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(count):,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 2. Missing rows breakdown
ax = axes[0, 1]
labels = ['Rows in Enriched\n(pre-reconciliation)', 'Rows added\n(from missing)', 'Total\n(reconciled)']
values = [len(df_enriched), len(df_missing), len(df_final)]
colors_pie = ['#ff7f0e', '#2ca02c', '#1f77b4']
wedges, texts, autotexts = ax.pie(values, labels=labels, autopct='%1.1f%%', colors=colors_pie, 
                                    startangle=90, textprops={'fontsize': 10})
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax.set_title('Dataset Composition', fontsize=12, fontweight='bold')

# 3. Diff coverage by source
ax = axes[1, 0]
enriched_with_diff = ((df_enriched['diff_text'].notna()) & (df_enriched['diff_text'] != '')).sum()
enriched_without_diff = len(df_enriched) - enriched_with_diff
missing_with_diff = ((df_missing['diff_text'].notna()) & (df_missing['diff_text'] != '')).sum()
missing_without_diff = len(df_missing) - missing_with_diff

sources = ['Pre-existing\nRows', 'Newly Added\nRows']
with_diff = [enriched_with_diff, missing_with_diff]
without_diff = [enriched_without_diff, missing_without_diff]

x = np.arange(len(sources))
width = 0.35
bars1 = ax.bar(x - width/2, with_diff, width, label='With Diffs', color='#2ca02c', alpha=0.7, edgecolor='black')
bars2 = ax.bar(x + width/2, without_diff, width, label='Without Diffs', color='#d62728', alpha=0.7, edgecolor='black')

ax.set_ylabel('Number of Rows', fontsize=11)
ax.set_title('Diff Extraction Status by Source', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(sources, fontsize=10)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{int(height):,}', ha='center', va='bottom', fontsize=9)

# 4. Success metrics
ax = axes[1, 1]
ax.axis('off')

# Create text summary
summary_text = f"""
RECONCILIATION RESULTS

✓ Missing Rows Identified:      {len(missing_commits):,} commits
✓ Successfully Added:             {len(df_final[df_final['commit_id'].isin(missing_commits)]):,} rows
✓ Diff Extraction Success:       {missing_with_diff:,} / {len(df_missing):,} ({100*missing_with_diff/len(df_missing):.1f}%)

FINAL DATASET

✓ Total Rows:                     {len(df_final):,}
✓ Unique Commits:                 {len(final_commits):,}
✓ Total Diff Coverage:            {n_with_diffs:,} / {n_total:,} ({coverage:.1f}%)
✓ All Original Commits Present:   {'YES ✓' if final_commits == all_should_have else 'NO ✗'}
✓ No Duplicate Commits:           {'YES ✓' if n_duplicates == 0 else 'NO ✗'}

OUTPUT FILE: {OUTPUT_CSV}
"""

ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.tight_layout()
plt.savefig('data_reconciliation_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Visualization saved as 'data_reconciliation_summary.png'")